In [11]:
import numpy as np
import pandas as pd
import time
import random
from tqdm import tqdm  
#For each action: Sample m futures
# For each sample:
# Get new belief + immediate reward
# Recursively estimate value of new belief (depth-1)
# Return: Maximum Q-value across all actions


#5 states
THETAS = np.array([-2, -1, 0, 1, 2]) 
 ##from gpt we should map states to -3-3, but if there are 45 states, it runs forever
NUM_STATES = len(THETAS)
QUESTION_COST = 1.0

#rasch IRT model 
def p_correct(theta, beta):
    return 1.0 / (1.0 + np.exp(-(theta - beta)))

#Bayesian Update
def update_belief(belief, beta, resp):
    likelihoods = p_correct(THETAS, beta)
    if resp == 1:
        post = belief * likelihoods
    else:
        post = belief * (1 - likelihoods) 

    s = post.sum()
    if s == 0:
        return np.ones(NUM_STATES) / NUM_STATES
    return post / s

# Variance
def belief_var(b):
    mean = np.sum(b * THETAS)
    return np.sum(b * (THETAS - mean) ** 2)



# MDP transition for sparse sampling
class AdaptiveMDP:
    def __init__(self, diffs): 
        self.diffs = diffs
        self.A = list(range(len(diffs))) #action space = 45

    def TR(self, belief, a):
        beta = self.diffs[a]

        # Sample ability
        idx = np.random.choice(NUM_STATES, p=belief)
        theta = THETAS[idx]

        # Simulate response
        response = 1 if np.random.rand() < p_correct(theta, beta) else 0

        # Update belief
        new_belief = update_belief(belief, beta, response)

        # Reward = info gain - cost
        old_var = belief_var(belief)
        new_var = belief_var(new_belief)
        reward = (old_var - new_var) - QUESTION_COST

        return new_belief, reward

# Sparse sampling depth=1 only
def sparse_sampling(mdp, belief, m=3):
    best_val = -1e18
    best_action = None

    for a in mdp.A:
        total = 0
        for _ in range(m):
            new_b, r = mdp.TR(belief, a)
            total += r
        val = total / m

        if val > best_val:
            best_val = val
            best_action = a

    return best_action



#  Estimate difficulty
def estimate_difficulties(df):
    stats = df.groupby("item")["resp"].agg(["sum", "count"])
    diffs = {}
    for item, row in stats.iterrows():
        p = (row["sum"] + 1) / (row["count"] + 2) #avoid 0/0
        p = np.clip(p, 0.01, 0.99)
        diffs[item] = -np.log(p / (1 - p)) 
    return diffs



# True ability
def estimate_true_state(student_df):
    p = student_df["resp"].mean()
    p = np.clip(p, 0.01, 0.99)
    theta = np.log(p / (1 - p))
    idx = np.argmin(np.abs(theta - THETAS))
    return idx


def run_test(student_df, qdiffs, m=3, threshold=0.1, maxq=45):

    items = student_df["item"].values

    # if len(items) > 25:
    #     items = np.random.choice(items, 25, replace=False)

    diffs = np.array([qdiffs[it] for it in items])
    available = list(range(len(items)))

    belief = np.ones(NUM_STATES) / NUM_STATES 
    asked = []

    while len(asked) < maxq and belief_var(belief) > threshold and len(available) > 0:

        mdp = AdaptiveMDP(diffs[available])
        a = sparse_sampling(mdp, belief, m=m)

        idx = available[a]
        item = items[idx]
        asked.append(item)

        available.pop(a)

        resp = student_df.loc[student_df["item"] == item, "resp"].values[0]

        belief = update_belief(belief, qdiffs[item], resp)

    return {
        "n_questions": len(asked),
        "final_var": belief_var(belief),
        "pred_state": int(np.argmax(belief)) 
    }


def main():
    df = pd.read_csv("df_small.csv")

    qdiffs = estimate_difficulties(df)

    results = []
    true_states = []
    pred_states = []

    start = time.time()

    # PROGRESS BAR HERE
    for sid in tqdm(df["id"].unique(), desc="Processing students"):
        sdf = df[df["id"] == sid]

        out = run_test(sdf, qdiffs, m=3)
        out["id"] = sid

        ts = estimate_true_state(sdf)
        out["true_state"] = ts

        true_states.append(ts)
        pred_states.append(out["pred_state"])

        results.append(out)

    elapsed = time.time() - start

    dfres = pd.DataFrame(results)
    accuracy = np.mean(np.array(true_states) == np.array(pred_states))
    print("\n=== SUMMARY ===")
    print("Students:", len(dfres))
    print("Avg questions:", dfres["n_questions"].mean())
    print("Accuracy:", f"{accuracy*100:.2f}%")
    print("Runtime:", elapsed, "sec")
    return dfres


if __name__ == "__main__":
    random.seed(0)
    np.random.seed(0)
    res = main()


Processing students: 100%|██████████| 5/5 [00:02<00:00,  2.30it/s]


=== SUMMARY ===
Students: 5
Avg questions: 38.8
Accuracy: 80.00%
Runtime: 2.183166980743408 sec


In [ ]:
import numpy as np
import pandas as pd
import time
import random
from tqdm import tqdm



THETAS = np.array([-2, 1, 0, 1, 2])
NUM_STATES = len(THETAS)
QUESTION_COST = 1.0

def p_correct(theta, beta):
    return 1.0 / (1.0 + np.exp(-(theta - beta)))

def update_belief(belief, beta, resp):
    likelihoods = p_correct(THETAS, beta)

    if resp == 1:
        post = belief * likelihoods 
    else:
        post = belief * (1 - likelihoods)

    s = post.sum()
    if s == 0:
        return np.ones(NUM_STATES) / NUM_STATES

    return post / s



# Belief variance
def belief_var(b):
    mean = np.sum(b * THETAS)
    return np.sum(b * (THETAS - mean) ** 2)


class AdaptiveMDP:
    def __init__(self, diffs):
        self.diffs = diffs
        self.A = list(range(len(diffs)))

    def TR(self, belief, a):
        beta = self.diffs[a]

        # sample true theta from current belief
        idx = np.random.choice(NUM_STATES, p=belief)
        theta = THETAS[idx]

        # sample response
        resp = 1 if np.random.rand() < p_correct(theta, beta) else 0

        # update posterior
        new_belief = update_belief(belief, beta, resp)

        # reward = info gain - cost
        r = belief_var(belief) - belief_var(new_belief) - QUESTION_COST

        return new_belief, r



def sparse_value(mdp, belief, depth, m=5, gamma=0.95):
    """
    Return an approximate value V(belief) using sparse sampling.

    depth = planning horizon (>= 0)
    m     = number of samples per action
    gamma = discount
    """
    if depth == 0:
        return 0.0

    best_q = -1e18

    for a in mdp.A:
        total = 0.0
        for _ in range(m):
            new_b, r = mdp.TR(belief, a)
            # recurse one level down
            total += r + gamma * sparse_value(mdp, new_b, depth - 1, m=m, gamma=gamma)

        q = total / m
        if q > best_q:
            best_q = q

    return float(best_q)


def sparse_action(mdp, belief, depth, m=3, gamma=0.95):
    """
    Choose the best action at this belief using sparse sampling
    with given planning depth.

    Returns: best action index (int)
    """
    assert depth >= 1, "depth must be >= 1 for action selection"

    best_q = -1e18
    best_a = None

    for a in mdp.A:
        total = 0.0
        for _ in range(m):
            new_b, r = mdp.TR(belief, a)
            total += r + gamma * sparse_value(mdp, new_b, depth - 1, m=m, gamma=gamma)

        q = total / m
        if q > best_q:
            best_q = q
            best_a = a

    return int(best_a)


def estimate_difficulties(df):
    stats = df.groupby("item")["resp"].agg(["sum", "count"])
    diffs = {}

    for item, row in stats.iterrows():
        # smoothed proportion correct
        p = (row["sum"] + 1) / (row["count"] + 2)
        p = np.clip(p, 0.01, 0.99)
        diffs[item] = -np.log(p / (1 - p))  # Rasch difficulty

    return diffs


def estimate_true_state(student_df):
    p = student_df["resp"].mean()
    p = np.clip(p, 0.01, 0.99)
    theta = np.log(p / (1 - p))
    idx = np.argmin(np.abs(theta - THETAS))
    return idx

def run_test(student_df, qdiffs, m=3, planning_depth=2,
             threshold=0.1, maxq=45, gamma=0.95):

    # list of items this student saw
    items = student_df["item"].values

    # if len(items) > 30:
    #     items = np.random.choice(items, 30, replace=False)

    diffs = np.array([qdiffs[it] for it in items])
    available = list(range(len(items)))

    belief = np.ones(NUM_STATES) / NUM_STATES
    asked_items = []

    while (
        len(asked_items) < maxq
        and belief_var(belief) > threshold
        and len(available) > 0
    ):

        mdp = AdaptiveMDP(diffs[available])
        a = sparse_action(mdp, belief, depth=planning_depth, m=m, gamma=gamma)

        idx = available[a]
        item = items[idx]
        asked_items.append(item)

        # remove this index from available
        available.pop(a)

        # look up the actual observed response
        resp = student_df.loc[student_df["item"] == item, "resp"].values[0]
        belief = update_belief(belief, qdiffs[item], resp)

    return {
        "n_questions": len(asked_items),
        "final_var": belief_var(belief),
        "pred_state": int(np.argmax(belief)),
        "asked_items": asked_items,
    }


def main():
    df = pd.read_csv("df_medium.csv")

    print("Estimating question difficulties...")
    qdiffs = estimate_difficulties(df)

    results = []
    true_states = []
    pred_states = []

    start = time.time()

    print("Running adaptive tests...")
    for sid in tqdm(df["id"].unique(), desc="Students processed"):
        sdf = df[df["id"] == sid]

        out = run_test(sdf, qdiffs, m=3, planning_depth=2, threshold=0.1, maxq=45)
        out["id"] = sid

        ts = estimate_true_state(sdf)
        out["true_state"] = ts

        true_states.append(ts)
        pred_states.append(out["pred_state"])
        results.append(out)

    elapsed = time.time() - start

    dfres = pd.DataFrame(results)
    accuracy = np.mean(np.array(true_states) == np.array(pred_states))

    print("\n================ SUMMARY ================")
    print("Students:", len(dfres))
    print("Avg questions asked:", dfres["n_questions"].mean())
    print("Estimated accuracy:", f"{accuracy * 100:.2f}%")
    print("Runtime:", f"{elapsed:.2f} sec")
    print("==========================================\n")

    return dfres


if __name__ == "__main__":
    random.seed(0)
    np.random.seed(0)
    res = main()


Estimating question difficulties...
Running adaptive tests...


Students processed: 100%|██████████| 100/100 [1:02:30<00:00, 37.51s/it]


================ SUMMARY ================
Students: 100
Avg questions asked: 27.81
Estimated accuracy: 79.00%
Runtime: 3750.99 sec



In [13]:
import numpy as np
import pandas as pd
import time
import random
from tqdm import tqdm



THETAS = np.array([-2, 1, 0, 1, 2])
NUM_STATES = len(THETAS)
QUESTION_COST = 1.0

def p_correct(theta, beta):
    return 1.0 / (1.0 + np.exp(-(theta - beta)))

def update_belief(belief, beta, resp):
    likelihoods = p_correct(THETAS, beta)

    if resp == 1:
        post = belief * likelihoods 
    else:
        post = belief * (1 - likelihoods)

    s = post.sum()
    if s == 0:
        return np.ones(NUM_STATES) / NUM_STATES

    return post / s



# Belief variance
def belief_var(b):
    mean = np.sum(b * THETAS)
    return np.sum(b * (THETAS - mean) ** 2)


class AdaptiveMDP:
    def __init__(self, diffs):
        self.diffs = diffs
        self.A = list(range(len(diffs)))

    def TR(self, belief, a):
        beta = self.diffs[a]

        # sample true theta from current belief
        idx = np.random.choice(NUM_STATES, p=belief)
        theta = THETAS[idx]

        # sample response
        resp = 1 if np.random.rand() < p_correct(theta, beta) else 0

        # update posterior
        new_belief = update_belief(belief, beta, resp)

        # reward = info gain - cost
        r = belief_var(belief) - belief_var(new_belief) - QUESTION_COST

        return new_belief, r



def sparse_value(mdp, belief, depth, m=3, gamma=0.95):
    """
    Return an approximate value V(belief) using sparse sampling.

    depth = planning horizon (>= 0)
    m     = number of samples per action
    gamma = discount
    """
    if depth == 0:
        return 0.0

    best_q = -1e18

    for a in mdp.A:
        total = 0.0
        for _ in range(m):
            new_b, r = mdp.TR(belief, a)
            # recurse one level down
            total += r + gamma * sparse_value(mdp, new_b, depth - 1, m=m, gamma=gamma)

        q = total / m
        if q > best_q:
            best_q = q

    return float(best_q)


def sparse_action(mdp, belief, depth, m=3, gamma=0.95):
    """
    Choose the best action at this belief using sparse sampling
    with given planning depth.

    Returns: best action index (int)
    """
    assert depth >= 1, "depth must be >= 1 for action selection"

    best_q = -1e18
    best_a = None

    for a in mdp.A:
        total = 0.0
        for _ in range(m):
            new_b, r = mdp.TR(belief, a)
            total += r + gamma * sparse_value(mdp, new_b, depth - 1, m=m, gamma=gamma)

        q = total / m
        if q > best_q:
            best_q = q
            best_a = a

    return int(best_a)


def estimate_difficulties(df):
    stats = df.groupby("item")["resp"].agg(["sum", "count"])
    diffs = {}

    for item, row in stats.iterrows():
        # smoothed proportion correct
        p = (row["sum"] + 1) / (row["count"] + 2)
        p = np.clip(p, 0.01, 0.99)
        diffs[item] = -np.log(p / (1 - p))  # Rasch difficulty

    return diffs


def estimate_true_state(student_df):
    p = student_df["resp"].mean()
    p = np.clip(p, 0.01, 0.99)
    theta = np.log(p / (1 - p))
    idx = np.argmin(np.abs(theta - THETAS))
    return idx

def run_test(student_df, qdiffs, m=3, planning_depth=2,
             threshold=0.1, maxq=45, gamma=0.95):

    # list of items this student saw
    items = student_df["item"].values

    # if len(items) > 30:
    #     items = np.random.choice(items, 30, replace=False)

    diffs = np.array([qdiffs[it] for it in items])
    available = list(range(len(items)))

    belief = np.ones(NUM_STATES) / NUM_STATES
    asked_items = []

    while (
        len(asked_items) < maxq
        and belief_var(belief) > threshold
        and len(available) > 0
    ):

        mdp = AdaptiveMDP(diffs[available])
        a = sparse_action(mdp, belief, depth=planning_depth, m=m, gamma=gamma)

        idx = available[a]
        item = items[idx]
        asked_items.append(item)

        # remove this index from available
        available.pop(a)

        # look up the actual observed response
        resp = student_df.loc[student_df["item"] == item, "resp"].values[0]
        belief = update_belief(belief, qdiffs[item], resp)

    return {
        "n_questions": len(asked_items),
        "final_var": belief_var(belief),
        "pred_state": int(np.argmax(belief)),
        "asked_items": asked_items,
    }


def main():
    df = pd.read_csv("df_medium.csv")

    print("Estimating question difficulties...")
    qdiffs = estimate_difficulties(df)

    results = []
    true_states = []
    pred_states = []

    start = time.time()

    print("Running adaptive tests...")
    for sid in tqdm(df["id"].unique(), desc="Students processed"):
        sdf = df[df["id"] == sid]

        out = run_test(sdf, qdiffs, m=3, planning_depth=2, threshold=0.1, maxq=45)
        out["id"] = sid

        ts = estimate_true_state(sdf)
        out["true_state"] = ts

        true_states.append(ts)
        pred_states.append(out["pred_state"])
        results.append(out)

    elapsed = time.time() - start

    dfres = pd.DataFrame(results)
    accuracy = np.mean(np.array(true_states) == np.array(pred_states))

    print("\n================ SUMMARY ================")
    print("Students:", len(dfres))
    print("Avg questions asked:", dfres["n_questions"].mean())
    print("Estimated accuracy:", f"{accuracy * 100:.2f}%")
    print("Runtime:", f"{elapsed:.2f} sec")
    print("==========================================\n")

    return dfres


if __name__ == "__main__":
    random.seed(0)
    np.random.seed(0)
    res = main()


Estimating question difficulties...
Running adaptive tests...


Students processed: 100%|██████████| 100/100 [34:22<00:00, 20.62s/it]


================ SUMMARY ================
Students: 100
Avg questions asked: 27.81
Estimated accuracy: 79.00%
Runtime: 2062.37 sec

